# Homework: estimating intervals for $\beta$ and $\gamma$

In the main Session 7 notebook, we estimated point values for the SIR rates from the synthetic testing records. Here we extend that work by asking how much those estimates could vary because we observed samples rather than the whole population.

We will build a 95% interval for $\gamma$ from the four-week follow-up sample, then use a parametric bootstrap to estimate an interval for $\beta$ from weekly surveillance counts. These ranges are conditional on the SIR model and the testing assumptions in the lesson.

## 1. Load the same testing records

The CSV has one row per completed test. We will use the same synthetic file as the main notebook and keep the population denominator and weekly sampling structure consistent with that analysis.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import binom, beta as beta_distribution

POPULATION = 10_000
TESTS_URL = "https://raw.githubusercontent.com/DACSS-Complexity/s7_epidemics/main/session7_synthetic_tests.csv"
records = pd.read_csv(TESTS_URL, parse_dates=["date"])
records.head()

## 2. Match each follow-up to its earlier positive test

For the recovery estimate, use only follow-up tests that occurred four weeks after a person’s first positive surveillance test. A follow-up gives status at that later time; it does not reveal the exact week recovery occurred.

In [ ]:
FOLLOWUP_WEEKS = 4

first_positive = records.loc[
    records.test_type.eq("surveillance") & records.result.eq("positive"),
    ["person_id", "week"],
].sort_values("week").drop_duplicates("person_id")
first_positive = first_positive.rename(columns={"week": "positive_week"})

followups = records.loc[records.test_type.eq("followup")].rename(
    columns={"week": "followup_week"}
)
returned = followups.merge(first_positive, on="person_id", how="inner")
returned = returned.loc[
    returned.followup_week - returned.positive_week == FOLLOWUP_WEEKS
].copy()
returned.result.value_counts()

## 3. Build an interval for $\gamma$

Let $p$ be the share of matched follow-ups that are still positive. Under the constant recovery-rate assumption, $p=e^{-\gamma d}$, so the point estimate is $\hat{\gamma}=-\ln(p)/d$.

First calculate the point estimate. Then we will form an exact binomial interval for $p$ and transform its endpoints into an interval for $\gamma$.

In [ ]:
followup_positives = int(returned.result.eq("positive").sum())
followup_total = len(returned)
still_positive_share = followup_positives / followup_total
GAMMA_ESTIMATE = -np.log(still_positive_share) / FOLLOWUP_WEEKS

print(f"Matched follow-ups: {followup_total}")
print(f"Still positive: {followup_positives} ({still_positive_share:.1%})")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")

For a binomial proportion, the exact 95% interval can be calculated with beta-distribution quantiles. Transform the interval for the still-positive share using $-\ln(p)/d$; because this transformation decreases as $p$ increases, reverse the endpoints.

In [ ]:
positive_share_lower = beta_distribution.ppf(
    0.025, followup_positives, followup_total - followup_positives + 1
)
positive_share_upper = beta_distribution.ppf(
    0.975, followup_positives + 1, followup_total - followup_positives
)

gamma_lower = -np.log(positive_share_upper) / FOLLOWUP_WEEKS
gamma_upper = -np.log(positive_share_lower) / FOLLOWUP_WEEKS
print(f"95% interval for gamma: {gamma_lower:.3f}–{gamma_upper:.3f} per week")

## 4. Prepare the weekly surveillance observations

Now use the random surveillance sample, not follow-ups. Keep the number of tests and positive tests for each week from week 2 onward. Week 1 is the pre-outbreak baseline in this synthetic dataset.

In [ ]:
surveillance = records.loc[records.test_type.eq("surveillance")].copy()
surveillance["positive"] = surveillance.result.eq("positive").astype(int)
weekly = surveillance.groupby("week").agg(
    tests=("test_id", "size"),
    positives=("positive", "sum"),
)
observed = weekly.loc[weekly.index >= 2]
observed_weeks = observed.index.to_numpy(dtype=int)
observed_n = observed.tests.to_numpy(dtype=int)
observed_positive = observed.positives.to_numpy(dtype=int)

weekly.head()

## 5. Recreate the fitted SIR observation model

The weekly update below uses the same discrete SIR calculation as the main notebook. For each observed week, it returns the model’s infectious count; dividing by population gives the probability of a positive surveillance test under the model.

In [ ]:
def sir_one_week(susceptible, infectious, recovered, beta, gamma):
    new_infections = susceptible * (-np.expm1(-beta * infectious / POPULATION))
    new_recoveries = infectious * (-np.expm1(-gamma))
    susceptible -= new_infections
    infectious += new_infections - new_recoveries
    recovered += new_recoveries
    return susceptible, infectious, recovered


def sir_states_by_week(beta, initial_infectious, gamma):
    susceptible = POPULATION - initial_infectious
    infectious = initial_infectious
    recovered = 0.0
    states = []
    previous_week = int(observed_weeks[0])

    for week in observed_weeks:
        for _ in range(int(week - previous_week)):
            susceptible, infectious, recovered = sir_one_week(
                susceptible, infectious, recovered, beta, gamma
            )
        states.append((susceptible, infectious, recovered))
        previous_week = int(week)

    return np.asarray(states)

For a candidate $\beta$ and starting infectious count, compare the model’s positive-test probabilities with the observed positive counts. The negative log-likelihood is a mismatch score: smaller values mean the candidate better matches the weekly counts.

In [ ]:
def negative_log_likelihood(log_parameters, positive_counts, gamma):
    beta, initial_infectious = np.exp(log_parameters)
    states = sir_states_by_week(beta, initial_infectious, gamma)
    positive_share = np.clip(states[:, 1] / POPULATION, 1e-12, 1 - 1e-12)
    return -binom.logpmf(positive_counts, observed_n, positive_share).sum()

Fit the surveillance observations using the estimated $\gamma$. The optimizer estimates $\beta$ and the infectious count at the start of week 2 together, just as in the main notebook.

In [ ]:
fit_bounds = [
    (np.log(0.001), np.log(2.0)),
    (np.log(0.1), np.log(POPULATION - 1)),
]
fit = minimize(
    negative_log_likelihood,
    x0=np.log([0.5, 20]),
    args=(observed_positive, GAMMA_ESTIMATE),
    method="L-BFGS-B",
    bounds=fit_bounds,
)
BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE = np.exp(fit.x)
print(f"Estimated beta: {BETA_ESTIMATE:.3f} per week")
print(f"Estimated infectious people at start of week 2: {INITIAL_INFECTIOUS_ESTIMATE:.0f}")

## 6. Use a parametric bootstrap for $\beta$

A parametric bootstrap asks: if the fitted model described the process, how much could the estimate vary across new samples of the same size? First calculate the fitted weekly positive shares and set a random seed so results are reproducible.

In [ ]:
fitted_positive_share = np.clip(
    sir_states_by_week(
        BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE, GAMMA_ESTIMATE
    )[:, 1] / POPULATION,
    1e-12, 1 - 1e-12,
)
rng = np.random.default_rng(7)
BOOTSTRAP_REPETITIONS = 200
bootstrap_beta = []

For each repetition, simulate positive counts using the original weekly test totals, then refit $\beta$ and the starting infectious count. $\gamma$ stays fixed at its point estimate here, so the resulting $\beta$ interval is conditional on that estimate.

In [ ]:
for _ in range(BOOTSTRAP_REPETITIONS):
    simulated_positive = rng.binomial(observed_n, fitted_positive_share)
    bootstrap_fit = minimize(
        negative_log_likelihood,
        x0=fit.x,
        args=(simulated_positive, GAMMA_ESTIMATE),
        method="L-BFGS-B",
        bounds=fit_bounds,
    )
    if bootstrap_fit.success:
        bootstrap_beta.append(np.exp(bootstrap_fit.x[0]))

beta_lower, beta_upper = np.percentile(bootstrap_beta, [2.5, 97.5])
print(f"Successful bootstrap fits: {len(bootstrap_beta)} of {BOOTSTRAP_REPETITIONS}")
print(f"95% interval for beta: {beta_lower:.3f}–{beta_upper:.3f} per week")

## Interpreting the intervals

These are **model-based sampling intervals**, not guaranteed bounds on the pathogen’s true rates. The $\beta$ interval is conditional on the fitted $\gamma$; the separate intervals do not form a joint 95% region. Use them as evidence-informed ranges for exploration, and state these limitations when interpreting the ABM.

### Homework discussion

1. Compare each point estimate with its interval. What does the interval add?
2. Use values within the data-based intervals in your ABM exploration. How do the peak infectious count and total infected share change?
3. Why should the independent endpoints of the $\beta$ and $\gamma$ intervals not automatically be described as a joint 95% range?